# Getting on the Machine · your first hour on Polaris

This is Lab 00. By the end of it your notebook will submit a job to Polaris, that job will run on a real compute node, and its output will be back in your `~/lab00/` folder — with nothing typed into a terminal. Every later lab reuses this exact pattern, so it is worth the hour.

**You will:**
1. Prove you can `ssh` to Polaris from this Hub without a password.
2. Learn where things live on the cluster (home vs `/eagle` project vs `/local/scratch`).
3. Load a compiler with the `module` system and build a tiny C program on the login node.
4. Submit a **batch job** with `qsub`, watch it queue and run, and read its output back.
5. Do the same thing **interactively** with `qsub -I` — the debug workflow you will use all semester.

**Nothing here uses a GPU or MPI yet.** Those come in labs 05 and 08. Today is just "how do I get code onto Polaris and get an answer back?"


## How this notebook works · Hub cells vs cluster cells

You are reading this notebook on a **Jupyter Hub that is not Polaris**. There are two places code can run in this lab:

| Where | How it looks in the notebook | What it can do |
|---|---|---|
| **Hub** (this Jupyter kernel) | plain Python or `!command` | drive ssh/scp, run analysis, plot |
| **Polaris login node** | `sshRun("...")` | edit files, compile, submit jobs |
| **Polaris compute node** | inside a job script `submitJob(...)` starts | run your program |

Every cell below says which of the three it targets. When you eventually work in the terminal (**File → New → Terminal** in JupyterLab), you can `source ~/lab00/labEnv.sh` and your shell will know the same host, user, and paths this notebook does.


In [ ]:
# [Hub] Load the shared lab toolkit (labHelpers.py ships in the course repo next to this notebook).
# It gives you preflight/checkpoint checks, sshRun/sshPut/sshGet, submitJob/waitJob, and
# the plotting primitives every later lab reuses.
from labHelpers import *


### Set up this lab's identity

`setupLab()` records which cluster you are targeting, which account (allocation) charges the job, which queue to submit to, and where your remote scratch lives. It exports those as environment variables that `sshRun`, `submitJob`, and every terminal shell you open will see. **Edit the values below to match your account:**


In [ ]:
# [Hub] Edit HPC_USER and HPC_PROJECT for your account, then run this cell.
env = setupLab(
    labName    = "lab00",
    host       = "polaris",              # ssh alias from your ~/.ssh/config
    remoteUser = os.environ.get("HPC_USER", "CHANGE_ME"),
    project    = os.environ.get("HPC_PROJECT", "CHANGE_ME"),
    queue      = "debug",                 # 'debug' is the small, short-turnaround queue
    scratch    = f"/eagle/{os.environ.get('HPC_PROJECT', 'CHANGE_ME')}/{os.environ.get('HPC_USER', 'CHANGE_ME')}",
)


### Preflight · check your environment

These checks run before you touch the cluster and tell you exactly what to fix if any fail. The most common failure is the ssh key — the callout under a failed check tells you what to do.


In [ ]:
# [Hub] Environment health check.
preflight([
    check("ssh + scp on this Hub",
          lambda: (bool(shutil.which('ssh') and shutil.which('scp')),
                   f"ssh={shutil.which('ssh')}, scp={shutil.which('scp')}")),
    check("HPC_USER is set (not CHANGE_ME)",
          lambda: (os.environ.get('HPC_USER','CHANGE_ME') != 'CHANGE_ME',
                   os.environ.get('HPC_USER','(unset)')),
          hint="Edit the setupLab() cell above with your ALCF username and re-run it."),
    check("HPC_PROJECT is set (not CHANGE_ME)",
          lambda: (os.environ.get('HPC_PROJECT','CHANGE_ME') != 'CHANGE_ME',
                   os.environ.get('HPC_PROJECT','(unset)')),
          hint="Edit the setupLab() cell above with your allocation (e.g. UIC-HPC)."),
    check("passwordless ssh to Polaris", sshReachable(),
          hint="See Part 1 below - generate an ed25519 key on this Hub and add its public half to "
               "~/.ssh/authorized_keys on Polaris. This lab walks you through it."),
    check("scheduler answers on Polaris", schedulerAnswers(),
          hint="Log in via the terminal and check `qstat -Q` runs. If it does not, contact ALCF support."),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')), ('scratch', env.get('HPC_SCRATCH','?'))])


## Part 1 · Passwordless ssh from the Hub to Polaris

You need one long-lived key pair on this Hub. The **private half** stays here (never leaves this account). The **public half** goes into `~/.ssh/authorized_keys` on Polaris. After that, every `ssh polaris` from this Hub just works, no password prompt, forever.

> **🔀 On a Slurm system** (Crux, UIC ACER Extreme): identical steps. Only the host alias in your > `~/.ssh/config` changes.


**[Notebook cell]** Generate an ed25519 key pair on this Hub, no passphrase (this account is already gated by your Hub login). If a key already exists, this leaves it alone.


In [ ]:
# [Hub] Create an ed25519 keypair if you don't have one yet.
!test -f ~/.ssh/id_ed25519 && echo 'key already exists, skipping' \
    || ssh-keygen -q -t ed25519 -N '' -f ~/.ssh/id_ed25519 -C "$USER@hpcnotebook"
!ls -l ~/.ssh/id_ed25519*


**[Notebook cell]** Add a stanza for Polaris to your `~/.ssh/config` so every tool on this Hub — including this notebook — can just say `polaris` and mean `polaris.alcf.anl.gov`. This is why every helper in `labHelpers.py` takes `host='polaris'` as a short alias.


In [ ]:
# [Hub] Append a Polaris stanza to ~/.ssh/config (only once).
from pathlib import Path
sshConfig = Path.home() / ".ssh" / "config"
sshConfig.parent.mkdir(mode=0o700, exist_ok=True)
stanza = f"""
Host polaris
    HostName polaris.alcf.anl.gov
    User {os.environ['HPC_USER']}
    IdentityFile ~/.ssh/id_ed25519
    ServerAliveInterval 60
"""
existing = sshConfig.read_text() if sshConfig.exists() else ""
if "Host polaris" not in existing:
    sshConfig.write_text(existing + stanza)
    sshConfig.chmod(0o600)
    print("added Polaris stanza")
else:
    print("Polaris stanza already present, leaving alone")
showFile(sshConfig, language='text')


**[Terminal]** Open **File → New → Terminal** and run the two commands below to push your public key to Polaris. This is the ONLY step where you have to type your ALCF password — after this, nothing ever asks again.

```bash
# From a Hub terminal, ONE time:
ssh-copy-id polaris                        # prompts for your ALCF password once
ssh polaris 'echo hello from polaris'      # should print immediately, no prompt
```

When the second command prints `hello from polaris` with no prompt, come back here and run the checkpoint below.


In [ ]:
checkpoint("Part 1 - passwordless ssh to Polaris", [
    check("id_ed25519 exists on this Hub", fileExists("~/.ssh/id_ed25519")),
    check("~/.ssh/config has a 'polaris' host", fileContains("~/.ssh/config", "Host polaris")),
    check("ssh polaris echoes without a password", sshReachable(),
          hint="Run `ssh-copy-id polaris` in a Hub terminal, enter your ALCF password once, then re-run."),
])


## Part 2 · Where things live on Polaris

Every HPC site has (at least) three filesystems that trade off differently. Confusing them is the number-one source of "my job crashed but the code is fine" tickets. On Polaris:

| Path | Speed | Size | Use for |
|---|---|---|---|
| `~` (home) | slow | small (~50 GB) | source code, dotfiles, this repo |
| `/eagle/<project>/<user>/` | fast | huge (TB) | job outputs, datasets, checkpoints |
| `/local/scratch/` (per-node) | very fast | ~1 TB, per node | job-lifetime temp files, gone at exit |

**Rule you never break:** do not write big or hot data to `~`. It will slow the whole shared filesystem down for everyone. Every lab in this series stages its work under `$HPC_SCRATCH` (which `setupLab` set to `/eagle/<project>/<user>`).

> **🔀 On a Slurm system**: the names differ (`/scratch`, `/projects`, `/tmp`), the pattern is > the same. Substitute your site's fast filesystem for `/eagle`.


In [ ]:
# [Hub -> Polaris login] Look around. sshRun runs its argument on the Polaris login node.
out, _ = sshRun("pwd && whoami && groups && quota -s 2>/dev/null | head -20")
print(out)


In [ ]:
# [Hub -> Polaris login] Create your per-lab scratch directory (idempotent).
labDir = env['HPC_LAB_DIR']       # /eagle/<project>/<user>/lab00
sshRun(f"mkdir -p {labDir} && ls -ld {labDir}")[0].splitlines()[-1]


In [ ]:
checkpoint("Part 2 - filesystems and scratch", [
    check("HPC_SCRATCH is under /eagle",
          lambda: (env.get('HPC_SCRATCH','').startswith('/eagle'),
                   env.get('HPC_SCRATCH',''))),
    check("lab00 scratch dir exists on Polaris", remoteFileExists(env['HPC_LAB_DIR'])),
])


## Part 3 · Modules · pick a compiler

Polaris (like every HPC site) uses **environment modules** to switch between compilers, MPI implementations, and CUDA versions. `module avail` lists what is installed; `module load` puts one on your PATH; `module list` shows what is currently loaded.

For lab00 we just need a C compiler. Polaris ships GCC via the `PrgEnv-gnu` environment.


In [ ]:
# [Hub -> Polaris login] Poke the module system.
out, _ = sshRun("bash -lc 'module list 2>&1 && echo --- && module avail PrgEnv 2>&1 | head -20'")
print(out)


> **🔀 On a Slurm system**: `module` is not a scheduler thing — it comes from Lmod / > environment-modules and works identically everywhere. What differs is the *names* of the > modules (`gcc` vs `PrgEnv-gnu` vs `intel-oneapi`).


## Part 4 · Build a tiny C program on the login node

Two files, both written from the notebook, both `scp`'d to Polaris. The whole point of this part is to teach the loop: **edit locally → scp → compile remotely → run**. Every later lab reuses it.


In [ ]:
# [Hub] Write hello.c locally.
helloSource = '''\
#include <stdio.h>
#include <unistd.h>
int main(void) {
    char host[256];
    gethostname(host, sizeof host);
    printf("hello from Polaris compute node: %s\\n", host);
    return 0;
}
'''
local = Path(env['labDir']) / "hello.c"
local.write_text(helloSource)
showFile(local, language='c')


In [ ]:
# [Hub -> Polaris] Copy it up, then compile on the login node.
sshPut(str(local), env['HPC_LAB_DIR'] + "/hello.c")
remoteLab = env['HPC_LAB_DIR']
out, code_ = sshRun(f"bash -lc 'cd {remoteLab} && cc -O2 -Wall hello.c -o hello && ls -l hello'")
print(out)


In [ ]:
checkpoint("Part 4 - built hello on Polaris", [
    check("hello.c present on Polaris", remoteFileExists(env['HPC_LAB_DIR'] + "/hello.c")),
    check("hello binary present on Polaris", remoteFileExists(env['HPC_LAB_DIR'] + "/hello")),
])


## Part 5 · Submit your first batch job

On the login node you compile and edit. You **do not run compute there** — it is shared with every other user in the world. Real work goes through the scheduler.

A PBS job script is a shell script with `#PBS` directives at the top that tell the scheduler what resources you want. Here is the minimum one that runs `hello` on a compute node:


In [ ]:
# [Hub] Write a PBS job script that runs hello.
pbs = f'''\
#!/bin/bash
#PBS -N lab00Hello
#PBS -A {env['HPC_PROJECT']}
#PBS -q {env.get('HPC_QUEUE', 'debug')}
#PBS -l select=1:ncpus=1
#PBS -l walltime=00:05:00
#PBS -l filesystems=home:eagle
#PBS -j oe
#PBS -o {env['HPC_LAB_DIR']}/hello.out

cd {env['HPC_LAB_DIR']}
date
./hello
date
'''
jobScript = Path(env['labDir']) / "hello.pbs"
jobScript.write_text(pbs)
showFile(jobScript, language='bash')


> **🔀 On a Slurm system**, the same script becomes:
> ```bash
> #!/bin/bash
> #SBATCH --job-name=lab00Hello
> #SBATCH --account=<project>
> #SBATCH --partition=debug
> #SBATCH --nodes=1 --ntasks=1 --cpus-per-task=1
> #SBATCH --time=00:05:00
> #SBATCH --output=hello.out
> ```
> The `submitJob()` helper below picks the right submitter automatically, so this notebook > cell works either way.


In [ ]:
# [Hub -> Polaris] Ship the script up, submit it, and get a job id back.
sshPut(str(jobScript), env['HPC_LAB_DIR'] + "/hello.pbs")
jobId = submitJob(env['HPC_LAB_DIR'] + "/hello.pbs")
print("job id:", jobId)


In [ ]:
# [Hub] Watch it queue and run. The debug queue on Polaris is usually 5-15 minutes to start.
waitJob(jobId, pollSeconds=15, maxSeconds=1800)


In [ ]:
# [Hub -> Polaris] Fetch the output back to the Hub and show it.
sshGet(env['HPC_LAB_DIR'] + "/hello.out", str(Path(env['labDir']) / "hello.out"))
showFile(Path(env['labDir']) / "hello.out", language='text', title='hello.out')


In [ ]:
checkpoint("Part 5 - first batch job", [
    check("hello.pbs present on Polaris", remoteFileExists(env['HPC_LAB_DIR'] + "/hello.pbs")),
    check("hello.out was produced", fileExists(str(Path(env['labDir']) / "hello.out"))),
    check("hello.out contains a compute node hostname",
          fileContains(str(Path(env['labDir']) / "hello.out"), "hello from Polaris compute node")),
])


## Part 6 · Interactive session — the debug workflow you will use all semester

Batch jobs are what you use once your code works. When it does *not* work, you want a shell prompt **on a compute node** so you can rebuild and rerun in seconds. That is `qsub -I`.

You do not run `qsub -I` from a notebook cell — it holds the terminal open. Instead, open **File → New → Terminal**, `source ~/lab00/labEnv.sh` (so `$HPC_LAB_DIR` is set), then:

```bash
# In a Hub terminal, one time to try it:
ssh polaris
# then, on the Polaris login node:
qsub -I -A $HPC_PROJECT -q debug -l select=1:ncpus=1 -l walltime=00:15:00 -l filesystems=home:eagle
# ... wait for the scheduler to give you a shell ...
cd $HPC_LAB_DIR
./hello
hostname                        # note: you're on a compute node, not a login node
exit                            # gives the allocation back
```

> **🔀 On a Slurm system**: `salloc --account=<project> --partition=debug --nodes=1 --time=00:15:00`

You will lean on this in every lab from lab03 on: `qsub -I` to iterate, `qsub` (batch) to produce results.


In [ ]:
# [Hub] Show what's in your queue now (should be empty if the hello job already finished).
qstatTable()


## Wrap up

You now have every piece the rest of the course leans on: a passwordless connection, a per-lab scratch dir on Polaris, a working compiler, a batch script pattern, and a scheduler that answers you. Every later lab starts the same way (`setupLab`, `preflight`, three checkpoints per part) and ends the same way (`labSummary`, `feedback`).


### Lab scorecard


In [ ]:
labSummary("Getting on the Machine")


---
### One-minute feedback

What worked, what didn't, what should be clearer. This is anonymous to your classmates and goes straight to the instructor.


In [ ]:
feedback("Getting on the Machine")
